# CXCR4 Molecular Docking - Publication-Grade Pipeline
### AutoDock Vina 1.2.5 | OpenBabel | RDKit ADMET | PLIP Interaction Profiling

> **Target:** CXCR4 (PDB: 3ODU) | **Box:** 30x30x30 A | **Exhaustiveness:** 32 | **Controls:** AMD3100, IT1t, Pirfenidone

**Run every cell in order. Do NOT skip.**
Estimated runtime: ~4-6 hours on Colab free CPU at exhaustiveness=32.

## CELL 1 - Install Tools and Verify Environment

In [ ]:
# ── AutoDock Vina 1.2.5 ──────────────────────────────────────────────────────
!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 \
     -O /usr/local/bin/vina
!chmod +x /usr/local/bin/vina

# ── OpenBabel (apt) ───────────────────────────────────────────────────────────
!apt-get install -y openbabel python3-openbabel 2>/dev/null | tail -3

# ── RDKit: correct package name on Colab (rdkit-pypi was renamed to rdkit) ───
import subprocess, sys
# Try rdkit first (new name), fallback to rdkit-pypi (old name), then conda
for pkg in ["rdkit", "rdkit-pypi"]:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       capture_output=True, text=True)
    if r.returncode == 0:
        try:
            import rdkit; print(f"rdkit installed via: {pkg}"); break
        except ImportError:
            continue
else:
    # Conda fallback (works on Colab)
    print("Trying conda install for rdkit...")
    !conda install -c conda-forge rdkit -y -q 2>/dev/null || \
     pip install -q "rdkit==2023.9.5" 2>/dev/null || \
     pip install -q rdkit-pypi 2>/dev/null || echo "rdkit install attempted via all methods"

# ── Other Python packages ─────────────────────────────────────────────────────
!pip install -q pandas numpy matplotlib seaborn

# ── PLIP ─────────────────────────────────────────────────────────────────────
!pip install -q plip 2>/dev/null || echo "PLIP install attempted"

# ── Verify all tools ──────────────────────────────────────────────────────────
import importlib
print("\n" + "="*55)
print("ENVIRONMENT CHECK")
print("="*55)

all_ok = True

# Vina check
r_vina = subprocess.run(["vina", "--version"], capture_output=True, text=True)
ok_vina = r_vina.returncode == 0
ver_vina = (r_vina.stdout + r_vina.stderr).split("\n")[0].strip()[:50]
print(f"  {'OK  ' if ok_vina else 'FAIL'} AutoDock Vina        {ver_vina}")
if not ok_vina: all_ok = False

# OpenBabel check — use python binding first, then CLI
obabel_ok = False
try:
    import openbabel; obabel_ok = True
    print(f"  OK   OpenBabel             (Python binding)")
except ImportError:
    r_ob = subprocess.run(["obabel", "--version"], capture_output=True, text=True)
    obabel_ok = r_ob.returncode == 0
    ver_ob = (r_ob.stdout + r_ob.stderr).split("\n")[0].strip()[:45]
    # OpenBabel prints version to stderr on some builds — check both
    if not obabel_ok:
        # obabel --version exits with code 1 on some versions but still works
        test = subprocess.run(["obabel", "-H"], capture_output=True, text=True)
        obabel_ok = True  # if it ran at all it's installed
        ver_ob = "Open Babel (installed)"
    print(f"  {'OK  ' if obabel_ok else 'FAIL'} OpenBabel             {ver_ob}")
if not obabel_ok: all_ok = False

# Python packages
for pkg in ["pandas", "numpy", "matplotlib", "seaborn"]:
    try:
        importlib.import_module(pkg)
        print(f"  OK   {pkg}")
    except ImportError:
        print(f"  FAIL {pkg}"); all_ok = False

# RDKit
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors
    print(f"  OK   rdkit")
except ImportError:
    print(f"  FAIL rdkit  <-- run: !pip install rdkit  then restart runtime")
    all_ok = False

# PLIP (optional — graceful if missing)
try:
    import plip; print(f"  OK   plip")
except ImportError:
    print(f"  WARN plip  (optional — Cell 11 will skip PLIP if absent)")

print("="*55)
if all_ok:
    print("\nALL TOOLS READY - proceed to Cell 2")
else:
    print("\nACTION REQUIRED:")
    print("  If rdkit FAIL: Runtime -> Restart runtime, then re-run Cell 1")
    print("  If OpenBabel FAIL: run  !apt-get install -y openbabel  separately")

## CELL 2 - Create Project Directory Structure

In [ ]:
import os
DIRS = ["receptor","ligands_sdf","ligands_pdbqt","docking_results",
        "poses_pdb","top10_poses","reports","reports/figures","plip_reports","pymol_scripts"]
for d in DIRS:
    os.makedirs(d, exist_ok=True)
    print(f"  {d}/")
print("\nProceed to Cell 3.")

## CELL 3 - Upload Pre-Cleaned Receptor PDB

Upload receptor cleaned in PyMOL:
- HOH removed
- Crystal ligand (IT1t) and all HETATM removed
- Protein ATOM records only

In [ ]:
from google.colab import files
import os

print("Upload your cleaned receptor .pdb file.")
uploaded = files.upload()
receptor_pdb = None
for filename, content in uploaded.items():
    if filename.lower().endswith(".pdb"):
        dest = f"receptor/{filename}"
        with open(dest, "wb") as f:
            f.write(content)
        receptor_pdb = dest
        print(f"\nSaved: {dest}  ({os.path.getsize(dest)/1024:.1f} KB)")
        with open(dest) as f:
            lines = f.readlines()
        atom_n   = sum(1 for l in lines if l.startswith("ATOM"))
        hetatm_n = sum(1 for l in lines if l.startswith("HETATM"))
        hoh_n    = sum(1 for l in lines if "HOH" in l)
        print(f"  ATOM:   {atom_n}")
        print(f"  HETATM: {hetatm_n}  {'WARNING - check for residual ligands' if hetatm_n else 'OK'}")
        print(f"  HOH:    {hoh_n}  {'WARNING - waters present' if hoh_n else 'OK'}")
    else:
        print(f"Skipped non-PDB: {filename}")
if receptor_pdb:
    with open("receptor/receptor_path.txt","w") as f:
        f.write(receptor_pdb)
    print("\nReceptor path saved. Proceed to Cell 4.")
else:
    print("\nERROR: No .pdb uploaded. Re-run this cell.")

## CELL 4 - Upload All Ligand SDF Files

Include controls:
- `AMD3100.sdf` (CID 65015) - positive control
- `IT1t.sdf` - crystal reference
- `Pirfenidone.sdf` - negative reference

In [ ]:
from google.colab import files
import os, shutil

print("Select ALL SDF files. Use Ctrl+A to select all at once.\n")
uploaded = files.upload()
sdf_count, skipped, controls_found = 0, [], []
CONTROLS = ["65015","amd3100","it1t","25154985","pirfenidone","5281040"]
for filename in uploaded.keys():
    if filename.lower().endswith(".sdf"):
        dest = f"ligands_sdf/{filename}"
        if os.path.exists(filename):
            shutil.move(filename, dest)
        sdf_count += 1
        for ctrl in CONTROLS:
            if ctrl.lower() in filename.lower():
                controls_found.append(filename)
    else:
        skipped.append(filename)
        if os.path.exists(filename): os.remove(filename)
total = len([f for f in os.listdir("ligands_sdf") if f.endswith(".sdf")])
print(f"Uploaded this session: {sdf_count}")
print(f"Total in folder:       {total}")
if skipped: print(f"Skipped: {skipped}")
if controls_found: print(f"\nControls detected: {controls_found}")
else: print("\nWARNING: No control files detected. Ensure AMD3100, IT1t, Pirfenidone are included.")
print("\nProceed to Cell 5.")

## CELL 5 - Ligand QC, Deduplication, and ADMET Descriptors

Uses RDKit to validate 3D coordinates, detect duplicates by canonical SMILES,
and compute MW, HBD, HBA, logP, TPSA, rotatable bonds, Lipinski RO5.

In [ ]:
import os, pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

sdf_files = sorted([f for f in os.listdir("ligands_sdf") if f.endswith(".sdf")])
print(f"Validating {len(sdf_files)} SDF files...\n")
records, failed_files, seen_smiles = [], [], {}

for sdf_file in sdf_files:
    path = f"ligands_sdf/{sdf_file}"
    cid  = sdf_file.replace(".sdf","").replace("Conformer3D_COMPOUND_CID_","").replace("Conformer3D_CID_","")
    try:
        supplier = Chem.SDMolSupplier(path, removeHs=False)
        mol = next((m for m in supplier if m is not None), None)
        if mol is None:
            failed_files.append((sdf_file,"RDKit could not parse")); continue
        positions = mol.GetConformer().GetPositions()
        has_3d = not all(p[2]==0.0 for p in positions)
        mol_nh = Chem.RemoveHs(mol)
        smi = Chem.MolToSmiles(mol_nh, isomericSmiles=True)
        is_dup = smi in seen_smiles
        dup_of = seen_smiles[smi] if is_dup else None
        if not is_dup: seen_smiles[smi] = cid
        mw   = Descriptors.ExactMolWt(mol_nh)
        hbd  = rdMolDescriptors.CalcNumHBD(mol_nh)
        hba  = rdMolDescriptors.CalcNumHBA(mol_nh)
        logp = Descriptors.MolLogP(mol_nh)
        tpsa = Descriptors.TPSA(mol_nh)
        rot  = rdMolDescriptors.CalcNumRotatableBonds(mol_nh)
        ro5v = sum([mw>500, hbd>5, hba>10, logp>5])
        records.append({"CID":cid,"Filename":sdf_file,"Has3D":has_3d,"IsDuplicate":is_dup,
            "DuplicateOf":dup_of,"MW":round(mw,2),"HBD":hbd,"HBA":hba,"LogP":round(logp,2),
            "TPSA":round(tpsa,2),"RotBonds":rot,"RO5_Violations":ro5v,
            "RO5_Pass":ro5v<=1,"SMILES":smi})
    except Exception as e:
        failed_files.append((sdf_file, str(e)))

df_qc = pd.DataFrame(records)
df_qc.to_csv("reports/ligand_QC.csv", index=False)
print("="*55 + "\nLIGAND QC REPORT\n" + "="*55)
print(f"  Total processed:     {len(sdf_files)}")
print(f"  Successfully parsed: {len(df_qc)}")
print(f"  Failed:              {len(failed_files)}")
print(f"  Have 3D coords:      {df_qc['Has3D'].sum()}")
print(f"  Missing 3D:          {(~df_qc['Has3D']).sum()}")
print(f"  Duplicates:          {df_qc['IsDuplicate'].sum()}")
print(f"  Pass Lipinski RO5:   {df_qc['RO5_Pass'].sum()}")
print("="*55)
if df_qc["IsDuplicate"].sum() > 0:
    dups = df_qc[df_qc["IsDuplicate"]][["CID","Filename","DuplicateOf"]]
    print("\nDUPLICATES (flagged in results):")
    print(dups.to_string(index=False))
if failed_files:
    print("\nFAILED FILES:")
    for fn, reason in failed_files: print(f"  {fn}: {reason}")
print("\nQC saved: reports/ligand_QC.csv\nProceed to Cell 6.")

## CELL 6 - Convert SDF to PDBQT (Publication-Grade)

- Physiological pH protonation (-p 7.4)
- Gasteiger charge verification
- gen3d fallback for flat/2D structures
- Atom count validation
- Complete failure log

In [ ]:
import os, subprocess, time, pandas as pd

sdf_files = sorted([f for f in os.listdir("ligands_sdf") if f.endswith(".sdf")])
try:
    df_qc = pd.read_csv("reports/ligand_QC.csv")
    needs_gen3d = set(df_qc[~df_qc["Has3D"]]["Filename"].tolist())
except Exception:
    needs_gen3d = set()

success, failed, warnings_list = [], [], []
start = time.time()
print(f"Converting {len(sdf_files)} SDF files to PDBQT\n")

for i, sdf_file in enumerate(sdf_files):
    cid = sdf_file.replace(".sdf","").replace("Conformer3D_COMPOUND_CID_","").replace("Conformer3D_CID_","")
    inp = f"ligands_sdf/{sdf_file}"
    out = f"ligands_pdbqt/{cid}.pdbqt"
    gen3d = ["--gen3d"] if sdf_file in needs_gen3d else []
    r = subprocess.run(["obabel", inp, "-O", out, "-h", "-p", "7.4"] + gen3d, capture_output=True, text=True)
    ok = False; fail_reason = ""
    if not os.path.exists(out) or os.path.getsize(out) == 0:
        fail_reason = "Empty output"
    else:
        with open(out) as f: lines = f.readlines()
        atom_lines = [l for l in lines if l.startswith("ATOM") or l.startswith("HETATM")]
        if len(atom_lines) < 3:
            fail_reason = f"Too few atoms ({len(atom_lines)})"
        else:
            charge_vals = []
            for l in atom_lines:
                parts = l.split()
                if len(parts) >= 12:
                    try: charge_vals.append(float(parts[11]))
                    except ValueError: pass
            if not charge_vals: fail_reason = "No Gasteiger charges found"
            else:
                if all(abs(c) < 1e-6 for c in charge_vals): warnings_list.append((cid,"All charges zero"))
                ok = True
    if not ok and not gen3d:
        r2 = subprocess.run(["obabel", inp, "-O", out, "-h", "-p", "7.4", "--gen3d"], capture_output=True, text=True)
        if os.path.exists(out) and os.path.getsize(out) > 0:
            with open(out) as f: lines2 = f.readlines()
            al2 = [l for l in lines2 if l.startswith("ATOM") or l.startswith("HETATM")]
            if len(al2) >= 3: ok = True; fail_reason = ""
    if ok:
        success.append(cid)
        if (i+1) % 15 == 0: print(f"  [{i+1:>3}/{len(sdf_files)}] OK  {cid}")
    else:
        failed.append((cid, fail_reason))
        print(f"  [{i+1:>3}/{len(sdf_files)}] FAIL {cid}: {fail_reason}")

pd.DataFrame({"CID":[c for c in success]+[c for c,_ in failed],
    "Status":["success"]*len(success)+["failed"]*len(failed),
    "Note":[""]*len(success)+[r for _,r in failed]}).to_csv("reports/conversion_report.csv", index=False)

elapsed = time.time()-start
print(f"\n{'='*55}\nCONVERSION SUMMARY\n{'='*55}")
print(f"  Successfully converted: {len(success)}/{len(sdf_files)}")
print(f"  Failed:                 {len(failed)}")
print(f"  Warnings:               {len(warnings_list)}")
print(f"  Time: {elapsed:.0f} seconds")
if failed:
    print("\nFailed (excluded from docking):")
    for cid, reason in failed: print(f"  {cid}: {reason}")
print("\nProceed to Cell 7.")

## CELL 7 - Receptor Preparation (PDB to PDBQT)

-xr = rigid receptor | -h = polar hydrogens | -p 7.4 = physiological pH protonation

Key CXCR4 residue check confirms correct receptor structure.

In [ ]:
import subprocess, os
from collections import defaultdict

try:
    with open("receptor/receptor_path.txt") as f: receptor_pdb = f.read().strip()
except FileNotFoundError:
    pdb_files = [f for f in os.listdir("receptor") if f.endswith(".pdb")]
    receptor_pdb = f"receptor/{pdb_files[0]}" if pdb_files else None

if not receptor_pdb or not os.path.exists(receptor_pdb):
    raise FileNotFoundError("No receptor PDB found. Run Cell 3.")

receptor_pdbqt = "receptor/receptor.pdbqt"
print(f"Input:  {receptor_pdb}\nOutput: {receptor_pdbqt}\n")

r = subprocess.run(["obabel", receptor_pdb, "-O", receptor_pdbqt, "-xr", "-h", "-p", "7.4"],
    capture_output=True, text=True)

if not os.path.exists(receptor_pdbqt) or os.path.getsize(receptor_pdbqt) == 0:
    print("ERROR: Receptor conversion failed!")
    print(r.stderr); raise RuntimeError("Receptor preparation failed")

with open(receptor_pdbqt) as f: lines = f.readlines()
atom_lines = [l for l in lines if l.startswith("ATOM")]
print(f"Receptor prepared successfully!")
print(f"  Input:  {os.path.getsize(receptor_pdb):,} bytes")
print(f"  Output: {os.path.getsize(receptor_pdbqt):,} bytes")
print(f"  ATOM records: {len(atom_lines)}")

res_found = defaultdict(list)
for l in atom_lines:
    parts = l.split()
    if len(parts) >= 6:
        resname, resnum = parts[3], parts[5]
        if resname in ("ASP","GLU","TRP","TYR","HIS") and resnum not in res_found[resname]:
            res_found[resname].append(resnum)
print("\n  Key binding site residue check:")
for res in ["ASP","GLU","TRP","TYR","HIS"]:
    print(f"    {res}: {len(res_found[res])} residues found")
print("\nProceed to Cell 8 (main docking - ~4-6 hrs at exhaustiveness=32).")

## CELL 8 - Run Publication-Grade Docking (Exhaustiveness=32)

**Configuration:**
- Box: 30 x 30 x 30 Angstrom centered on CXCR4 orthosteric site
- Exhaustiveness: **32** (publication standard; 4x default)
- Num modes: 9 (best + 8 alternatives for convergence analysis)
- CPU: all available cores auto-detected
- Crash recovery: checkpoint saves after every compound
- Robust affinity parsing: regex primary + VINA RESULT fallback

**Expected runtime:** 3-5 hours for 105 compounds on Colab free CPU.

In [ ]:
import subprocess, os, time, json, re
import pandas as pd

CENTER_X, CENTER_Y, CENTER_Z = 20.610, -7.972, 71.068
SIZE_X, SIZE_Y, SIZE_Z = 30, 30, 30
EXHAUSTIVENESS = 32
NUM_MODES = 9
CPU_COUNT = os.cpu_count() or 4
RECEPTOR  = "receptor/receptor.pdbqt"
VINA      = "/usr/local/bin/vina"
CHECKPOINT = "docking_results/.checkpoint.json"

CID_NAMES = {
    "2353":"Rutin","3564":"Baicalin","4788":"Berberine","6167":"Colchicine",
    "9064":"Emetine","10114":"EGCG","10219":"Quercetin","10281":"Kaempferol",
    "10364":"Luteolin","10494":"Naringenin","10621":"Apigenin","19009":"Hesperidin",
    "24360":"Camptothecin","34458":"Curcumin","40632":"Catechin","64945":"Icariin",
    "64971":"Puerarin","65015":"AMD3100 [POSITIVE CTRL]","65064":"Epicatechin",
    "65752":"Genistein","68071":"Daidzein","68827":"Formononetin","72276":"Chrysin",
    "72281":"Galangin","72300":"Fisetin","72301":"Wogonin","72303":"Baicalein",
    "72322":"Myricetin","72378":"Morin","73078":"Tanshinone IIA","82143":"Isorhamnetin",
    "91466":"Scutellarein","99474":"Taxifolin","101301":"Matrine","107985":"Oxymatrine",
    "119258":"Harmine","122724":"Harmaline","135423438":"Piperine",
    "1548943":"Withaferin A","160254":"Celastrol","164676":"Ursolic acid",
    "168928":"Oleanolic acid","265237":"Glycyrrhetinic acid","275182":"Betulinic acid",
    "275196":"Boswellic acid","439246":"Artesunate","439533":"Artemisinin",
    "440735":"Parthenolide","442088":"Honokiol","442534":"Magnolol",
    "637760":"Thymoquinone","638024":"Carvacrol","638278":"Capsaicin",
    "639665":"Bakuchiol","641785":"Zerumbone","969516":"Andrographolide",
    "5273755":"Cryptotanshinone","5280343":"Apigenin-2","5280373":"Kaempferol-2",
    "5280378":"Quercetin-2","5280441":"Luteolin-2","5280442":"Fisetin-2",
    "5280443":"Myricetin-2","5280445":"Morin-2","5280448":"Galangin-2",
    "5280459":"Chrysin-2","5280666":"Hesperetin","5280805":"Pinocembrin",
    "5280863":"Naringenin-2","5280953":"Eriodictyol","5280961":"Acacetin",
    "5281222":"Diosmetin","5281316":"Wogonin-2","5281605":"Eupatilin",
    "5281607":"Hispidulin","5281612":"Chrysoeriol","5281614":"Rhamnetin",
    "5281616":"Scutellarein-2","5281628":"Vitexin","5281643":"Orientin",
    "5281654":"Isoquercitrin","5281670":"Quercitrin","5281672":"Astilbin",
    "5281675":"Hyperoside","5281691":"Biochanin A","5281697":"Calycosin",
    "5281703":"Tectorigenin","5281708":"Formononetin-2","5281807":"Isoliquiritigenin",
    "5281811":"Butein","5316743":"Licochalcone A","5318517":"Xanthohumol",
    "5318997":"Cardamonin","5318998":"Echinatin","5321010":"Phloretin",
    "5459308":"Tetrandrine","5468522":"Berbamine","5470187":"Sinomenine",
    "5484006":"Noscapine","6442675":"Lycorine","6473881":"Triptolide",
    "6917864":"Dihydrotanshinone","9547215":"Oridonin","24864132":"Cucurbitacin B",
    "25147749":"Diosgenin","25154985":"IT1t [CRYSTAL REF]",
    "5281040":"Pirfenidone [NEGATIVE CTRL]",
}

completed = {}
if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT) as f: completed = json.load(f)
    print(f"Checkpoint: {len(completed)} compounds already done. Resuming...")

ligand_files = sorted([f for f in os.listdir("ligands_pdbqt") if f.endswith(".pdbqt")])
total = len(ligand_files)
print(f"Found {total} PDBQT ligands")
print(f"Config: center=({CENTER_X},{CENTER_Y},{CENTER_Z}), box={SIZE_X}x{SIZE_Y}x{SIZE_Z}, exhaustiveness={EXHAUSTIVENESS}, cpu={CPU_COUNT}\n")

start_time = time.time()
results = dict(completed)
error_log = {}

for i, lig_file in enumerate(ligand_files):
    cid  = lig_file.replace(".pdbqt","")
    name = CID_NAMES.get(cid, cid)
    if cid in completed:
        if (i+1) % 20 == 0: print(f"  [{i+1:>3}/{total}] SKIP {name[:35]} (done: {completed[cid]:.3f})")
        continue
    inp_lig  = f"ligands_pdbqt/{lig_file}"
    out_file = f"docking_results/{cid}_out.pdbqt"
    cmd = [VINA,"--receptor",RECEPTOR,"--ligand",inp_lig,"--out",out_file,
           "--center_x",str(CENTER_X),"--center_y",str(CENTER_Y),"--center_z",str(CENTER_Z),
           "--size_x",str(SIZE_X),"--size_y",str(SIZE_Y),"--size_z",str(SIZE_Z),
           "--exhaustiveness",str(EXHAUSTIVENESS),"--num_modes",str(NUM_MODES),"--cpu",str(CPU_COUNT)]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=900)
    except subprocess.TimeoutExpired:
        error_log[cid] = "timeout"
        print(f"  [{i+1:>3}/{total}] TIMEOUT {name[:35]}"); continue

    best_affinity = None
    for line in r.stdout.split("\n"):
        m = re.match(r"^\s+1\s+([-\d.]+)\s+", line)
        if m:
            try: best_affinity = float(m.group(1)); break
            except ValueError: pass
    if best_affinity is None and os.path.exists(out_file):
        with open(out_file) as f:
            for line in f:
                if "VINA RESULT" in line:
                    try: best_affinity = float(line.split()[3]); break
                    except (IndexError, ValueError): pass

    elapsed = time.time()-start_time
    done_count = i+1-len(completed)
    eta_sec = (elapsed/max(done_count,1))*(total-i-1)
    if best_affinity is not None:
        results[cid] = best_affinity
        with open(CHECKPOINT,"w") as f: json.dump(results, f)
        ctrl = " [CTRL]" if any(t in name for t in ["CTRL","REF"]) else ""
        print(f"  [{i+1:>3}/{total}] {best_affinity:>8.3f} kcal/mol  {name[:35]:<35}  ETA {eta_sec/60:.1f} min{ctrl}")
    else:
        error_log[cid] = r.stderr[:80].strip()
        print(f"  [{i+1:>3}/{total}] FAILED  {name[:35]}  ({r.stderr[:50].strip()})")

rows = [{"CID":cid,"Compound":CID_NAMES.get(cid,cid),"Best_Affinity_kcal_mol":aff,
         "IsControl":any(t in CID_NAMES.get(cid,"") for t in ["CTRL","REF"])}
        for cid,aff in results.items()]
df = pd.DataFrame(rows).sort_values("Best_Affinity_kcal_mol").reset_index(drop=True)
df["Rank"] = df.index + 1
df.to_csv("reports/docking_results_raw.csv", index=False)

total_time = time.time()-start_time
print(f"\n{'='*60}\nDOCKING COMPLETE in {total_time/3600:.2f} hours | {len(results)}/{total} compounds\n{'='*60}")
print("\nCONTROL PERFORMANCE:")
for _, row in df[df["IsControl"]].iterrows():
    print(f"  {str(row['Compound']):<45} {row['Best_Affinity_kcal_mol']:>8.3f} kcal/mol")
print("\nTOP 20 TEST COMPOUNDS:")
print(f"  {'Rank':<5} {'Compound':<38} {'CID':<14} {'Score':>8}")
print(f"  {'-'*65}")
for _, r in df[~df["IsControl"]].head(20).iterrows():
    print(f"  {int(r['Rank']):<5} {str(r['Compound']):<38} {str(r['CID']):<14} {r['Best_Affinity_kcal_mol']:>8.3f}")
print("\nResults saved: reports/docking_results_raw.csv\nProceed to Cell 9.")

## CELL 9 - Post-Processing: ADMET Filtering and Final Ranking

Merges docking scores with RDKit ADMET data, computes delta vs AMD3100,
classifies binding tiers, and produces the final publication-ready ranked table.

In [ ]:
import pandas as pd, os

df_dock = pd.read_csv("reports/docking_results_raw.csv")
df_qc   = pd.read_csv("reports/ligand_QC.csv")
df_dock["CID"] = df_dock["CID"].astype(str)
df_qc["CID"]   = df_qc["CID"].astype(str)

df = pd.merge(df_dock, df_qc[["CID","MW","HBD","HBA","LogP","TPSA","RotBonds",
    "RO5_Violations","RO5_Pass","IsDuplicate","DuplicateOf","SMILES"]], on="CID", how="left")

amd_rows = df[df["Compound"].str.contains("AMD3100", na=False)]
AMD3100_SCORE = amd_rows.iloc[0]["Best_Affinity_kcal_mol"] if len(amd_rows)>0 else -7.2
print(f"AMD3100 reference score: {AMD3100_SCORE:.3f} kcal/mol")

df["Delta_vs_AMD3100"]  = df["Best_Affinity_kcal_mol"] - AMD3100_SCORE
df["Beats_AMD3100"]     = df["Best_Affinity_kcal_mol"] < AMD3100_SCORE
df["Flag_HighMW"]       = df["MW"].fillna(0) > 500
df["Flag_HighTPSA"]     = df["TPSA"].fillna(0) > 140
df["Flag_HighRotBonds"] = df["RotBonds"].fillna(0) > 10

def binding_tier(s):
    if s<=-10.0: return "Tier 1 - Excellent"
    if s<=-9.0:  return "Tier 2 - Strong"
    if s<=-8.0:  return "Tier 3 - Good"
    if s<=-7.0:  return "Tier 4 - Moderate"
    return "Tier 5 - Weak"

df["Binding_Tier"] = df["Best_Affinity_kcal_mol"].apply(binding_tier)
df_ctrl = df[df["IsControl"]==True].copy()
df_test = df[df["IsControl"]!=True].copy().sort_values("Best_Affinity_kcal_mol").reset_index(drop=True)
df_test["Rank"] = df_test.index+1

df_test.to_csv("reports/final_ranked_compounds.csv", index=False)
df_ctrl.to_csv("reports/control_compounds.csv", index=False)
df.to_csv("reports/full_results_with_ADMET.csv", index=False)

print(f"\n{'='*65}\nFINAL RESULTS - CXCR4 (PDB: 3ODU)\n{'='*65}")
print(f"  Total test compounds:       {len(df_test)}")
print(f"  Beat AMD3100 threshold:     {df_test['Beats_AMD3100'].sum()}")
print(f"  Tier 1-2 (score <= -9.0):  {(df_test['Best_Affinity_kcal_mol']<=-9.0).sum()}")
if "RO5_Pass" in df_test: print(f"  Pass Lipinski RO5:          {df_test['RO5_Pass'].sum()}")
print("\n  CONTROLS:")
for _, r in df_ctrl.sort_values("Best_Affinity_kcal_mol").iterrows():
    print(f"    {str(r['Compound']):<45} {r['Best_Affinity_kcal_mol']:>8.3f} kcal/mol")
print("\n  TOP 20 TEST COMPOUNDS:")
print(f"  {'Rank':<5} {'Compound':<33} {'Score':>8}  {'MW':>6} {'LogP':>5} {'RO5':>4} {'Beats Ctrl':>10}  Tier")
print(f"  {'-'*85}")
for _, r in df_test.head(20).iterrows():
    ro5   = "YES" if r.get("RO5_Pass",True) else "NO "
    beats = "YES" if r.get("Beats_AMD3100",False) else "NO "
    print(f"  {int(r['Rank']):<5} {str(r['Compound']):<33} {r['Best_Affinity_kcal_mol']:>8.3f}"
          f"  {r.get('MW',0):>6.1f} {r.get('LogP',0):>5.2f} {ro5:>4}  {beats:>10}  {str(r['Binding_Tier'])[:18]}")
print("\nFiles saved: reports/final_ranked_compounds.csv\nProceed to Cell 10.")

## CELL 10 - Publication-Quality Figures

Generates 4 figures (PNG 300dpi + SVG vector):
1. Ranked affinity bar chart (top 30)
2. Score distribution histogram
3. MW vs Affinity scatter (ADMET context)
4. Binding tier pie chart

In [ ]:
import pandas as pd, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

os.makedirs("reports/figures", exist_ok=True)
df_test = pd.read_csv("reports/final_ranked_compounds.csv")
df_ctrl = pd.read_csv("reports/control_compounds.csv")
try:
    AMD3100_SCORE = df_ctrl[df_ctrl["Compound"].str.contains("AMD3100",na=False)].iloc[0]["Best_Affinity_kcal_mol"]
except Exception:
    AMD3100_SCORE = -7.2

PALETTE = {"Tier 1 - Excellent":"#b22222","Tier 2 - Strong":"#d94e00",
           "Tier 3 - Good":"#e07b00","Tier 4 - Moderate":"#6baed6","Tier 5 - Weak":"#bdbdbd"}

def save_fig(fig, name):
    fig.savefig(f"reports/figures/{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(f"reports/figures/{name}.svg", bbox_inches="tight")
    plt.close(fig); print(f"  Saved: {name}.png + .svg")

# Figure 1: Ranked bar chart
top30 = df_test.head(30).copy()
colors = [PALETTE.get(t,"#bdbdbd") for t in top30["Binding_Tier"]]
fig, ax = plt.subplots(figsize=(14,8))
bars = ax.barh(range(len(top30)), top30["Best_Affinity_kcal_mol"], color=colors, edgecolor="white", linewidth=0.5, height=0.75)
ax.set_yticks(range(len(top30))); ax.set_yticklabels(top30["Compound"], fontsize=8.5); ax.invert_yaxis()
ax.axvline(AMD3100_SCORE, color="#1a1a2e", linestyle="--", linewidth=2,
           label=f"AMD3100 (positive ctrl): {AMD3100_SCORE:.3f} kcal/mol")
ax.set_xlabel("Binding Affinity (kcal/mol)", fontsize=12, fontweight="bold")
ax.set_title("CXCR4 Molecular Docking - Top 30 Natural Compounds\nAutoDock Vina 1.2.5 | Exhaustiveness=32 | PDB: 3ODU",
             fontsize=12, fontweight="bold", pad=10)
ax.set_xlim(min(top30["Best_Affinity_kcal_mol"])-0.5, 0)
ax.grid(axis="x", alpha=0.3, linestyle=":")
patches = [mpatches.Patch(color=v,label=k) for k,v in PALETTE.items() if k in top30["Binding_Tier"].values]
ax.legend(handles=patches+ax.get_lines(), loc="lower right", fontsize=8.5, framealpha=0.9)
for i, (bar, val) in enumerate(zip(bars, top30["Best_Affinity_kcal_mol"])):
    ax.text(val-0.05, i, f"{val:.2f}", va="center", ha="right", fontsize=7, color="white", fontweight="bold")
plt.tight_layout(); save_fig(fig, "fig1_ranked_affinities")

# Figure 2: Score distribution
fig, ax = plt.subplots(figsize=(10,5))
n, bins, patches_h = ax.hist(df_test["Best_Affinity_kcal_mol"], bins=25, color="#4e79a7", edgecolor="white", linewidth=0.5, alpha=0.85)
for patch, left in zip(patches_h, bins):
    if left < AMD3100_SCORE: patch.set_facecolor("#b22222"); patch.set_alpha(0.9)
ax.axvline(AMD3100_SCORE, color="black", linestyle="--", linewidth=2, label=f"AMD3100: {AMD3100_SCORE:.3f} kcal/mol")
beats = (df_test["Best_Affinity_kcal_mol"] < AMD3100_SCORE).sum()
ax.set_xlabel("Binding Affinity (kcal/mol)", fontsize=12, fontweight="bold")
ax.set_ylabel("Number of Compounds", fontsize=12, fontweight="bold")
ax.set_title(f"Docking Score Distribution (n={len(df_test)} compounds)\nRed = better than AMD3100 ({beats} compounds)",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=10); ax.grid(alpha=0.3, linestyle=":")
plt.tight_layout(); save_fig(fig, "fig2_score_distribution")

# Figure 3: MW vs Affinity
if "MW" in df_test.columns and df_test["MW"].notna().sum() > 5:
    fig, ax = plt.subplots(figsize=(10,7))
    ro5_col = df_test.get("RO5_Pass", pd.Series([True]*len(df_test))).fillna(True)
    colors_s = ["#2c7bb6" if p else "#d7191c" for p in ro5_col]
    ax.scatter(df_test["MW"], df_test["Best_Affinity_kcal_mol"], c=colors_s, s=55, alpha=0.75, edgecolors="white", linewidth=0.4)
    ax.axhline(AMD3100_SCORE, color="black", linestyle="--", linewidth=1.5, label=f"AMD3100: {AMD3100_SCORE:.3f} kcal/mol")
    ax.axvline(500, color="gray", linestyle=":", linewidth=1.2, label="MW=500 (Lipinski)")
    ax.set_xlabel("Molecular Weight (Da)", fontsize=12, fontweight="bold")
    ax.set_ylabel("Binding Affinity (kcal/mol)", fontsize=12, fontweight="bold")
    ax.set_title("Molecular Weight vs. Binding Affinity\nBlue=Lipinski pass | Red=RO5 violation", fontsize=11, fontweight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3, linestyle=":")
    for _, r in df_test.head(5).iterrows():
        if pd.notna(r.get("MW")):
            ax.annotate(str(r["Compound"])[:18], (r["MW"], r["Best_Affinity_kcal_mol"]),
                textcoords="offset points", xytext=(5,3), fontsize=7,
                arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))
    plt.tight_layout(); save_fig(fig, "fig3_mw_vs_affinity")

# Figure 4: Binding tier pie
tier_counts = df_test["Binding_Tier"].value_counts()
tier_order = [t for t in PALETTE if t in tier_counts.index]
sizes_pie = [tier_counts[t] for t in tier_order]
colors_pie = [PALETTE[t] for t in tier_order]
fig, ax = plt.subplots(figsize=(8,6))
wedges, texts, autotexts = ax.pie(sizes_pie, labels=None, colors=colors_pie,
    autopct=lambda p: f"{p:.1f}%\n(n={int(round(p*sum(sizes_pie)/100))})" if p>2 else "",
    startangle=90, wedgeprops={"edgecolor":"white","linewidth":1.5}, pctdistance=0.75)
for at in autotexts: at.set_fontsize(9)
ax.legend(wedges, tier_order, title="Binding Tier", loc="lower right", fontsize=9, title_fontsize=10)
ax.set_title(f"Binding Tier Classification - CXCR4 Library (n={len(df_test)})", fontsize=11, fontweight="bold")
plt.tight_layout(); save_fig(fig, "fig4_binding_tiers")
print("\nAll 4 figures saved to reports/figures/\nProceed to Cell 11.")

## CELL 11 - Convert Best Poses to PDB and PLIP Interaction Analysis

Converts all docked poses to PDB, then runs PLIP on top 10 to formally profile
H-bonds, hydrophobic contacts, pi-stacking, and salt bridges.

In [ ]:
import subprocess, os, shutil, re, pandas as pd

df_ranked = pd.read_csv("reports/final_ranked_compounds.csv")
os.makedirs("poses_pdb", exist_ok=True)
os.makedirs("top10_poses", exist_ok=True)
os.makedirs("plip_reports", exist_ok=True)

out_files = sorted([f for f in os.listdir("docking_results") if f.endswith("_out.pdbqt")])
print(f"Converting {len(out_files)} poses to PDB...")
converted, failed_conv = [], []
for out_file in out_files:
    cid = out_file.replace("_out.pdbqt","")
    inp = f"docking_results/{out_file}"
    outp = f"poses_pdb/{cid}_best.pdb"
    r = subprocess.run(["obabel", inp, "-O", outp, "--firstmodel"], capture_output=True, text=True)
    if os.path.exists(outp) and os.path.getsize(outp)>0: converted.append(cid)
    else: failed_conv.append(cid)
print(f"Converted: {len(converted)}/{len(out_files)}")
if failed_conv: print(f"Failed: {failed_conv}")

print("\nCopying top 10 poses...")
top10_cids = df_ranked.head(10)["CID"].astype(str).tolist()
for cid in top10_cids:
    src = f"poses_pdb/{cid}_best.pdb"; dst = f"top10_poses/{cid}_top_pose.pdb"
    if os.path.exists(src):
        shutil.copy(src, dst)
        name = df_ranked[df_ranked["CID"].astype(str)==cid]["Compound"].values
        print(f"  OK: {name[0] if len(name)>0 else cid}")
    else: print(f"  MISSING: CID {cid}")

try:
    with open("receptor/receptor_path.txt") as f: receptor_pdb_path = f.read().strip()
except Exception:
    pdb_files = [f for f in os.listdir("receptor") if f.endswith(".pdb")]
    receptor_pdb_path = f"receptor/{pdb_files[0]}" if pdb_files else None

print("\nRunning PLIP interaction analysis on top 10...")
plip_summary = []
for cid in top10_cids:
    name_arr = df_ranked[df_ranked["CID"].astype(str)==str(cid)]["Compound"].values
    name = name_arr[0] if len(name_arr)>0 else cid
    pose_pdb = f"top10_poses/{cid}_top_pose.pdb"
    if not os.path.exists(pose_pdb):
        plip_summary.append({"CID":cid,"Compound":name,"PLIP_Status":"pose_missing"}); continue
    combined = f"plip_reports/{cid}_complex.pdb"
    try:
        if receptor_pdb_path and os.path.exists(receptor_pdb_path):
            with open(receptor_pdb_path) as f: rec_lines = [l for l in f if l.startswith("ATOM")]
            with open(pose_pdb) as f: lig_lines = f.readlines()
            lig_hetatm = []
            for l in lig_lines:
                if l.startswith("ATOM"): l = "HETATM" + l[6:]
                if l.startswith("ATOM") or l.startswith("HETATM"): lig_hetatm.append(l)
            with open(combined,"w") as out:
                out.writelines(rec_lines); out.write("TER\n"); out.writelines(lig_hetatm); out.write("END\n")
        else: shutil.copy(pose_pdb, combined)
    except Exception as e: combined = pose_pdb
    plip_out = f"plip_reports/{cid}"; os.makedirs(plip_out, exist_ok=True)
    r_plip = subprocess.run(["python","-m","plip.plipcmd","-f",combined,"-o",plip_out,"-t","-x"],
        capture_output=True, text=True, timeout=120)
    hb=hy=pi=sb=0
    if r_plip.returncode==0:
        txt_files = [f for f in os.listdir(plip_out) if f.endswith(".txt")]
        if txt_files:
            with open(f"{plip_out}/{txt_files[0]}") as f: content = f.read()
            for pattern, var in [(r"HYDROGEN BONDS\s*\|\s*(\d+)","hb"),(r"HYDROPHOBIC INTERACTIONS\s*\|\s*(\d+)","hy"),
                                  (r"PI-STACKING\s*\|\s*(\d+)","pi"),(r"SALT BRIDGES\s*\|\s*(\d+)","sb")]:
                m = re.search(pattern, content)
                if m:
                    v = int(m.group(1))
                    if var=="hb": hb=v
                    elif var=="hy": hy=v
                    elif var=="pi": pi=v
                    elif var=="sb": sb=v
        plip_summary.append({"CID":cid,"Compound":name,"HBonds":hb,"Hydrophobic":hy,"PiStacking":pi,"SaltBridges":sb,"PLIP_Status":"success"})
        print(f"  OK  {str(name)[:30]:<30} HB={hb} Hydrophob={hy} Pi={pi} Salt={sb}")
    else:
        plip_summary.append({"CID":cid,"Compound":name,"PLIP_Status":"failed-use-PyMOL"})
        print(f"  PLIP unavailable for {str(name)[:30]} - use PyMOL distance tool manually")

pd.DataFrame(plip_summary).to_csv("reports/plip_interaction_summary.csv", index=False)
print("\nPLIP summary: reports/plip_interaction_summary.csv\nProceed to Cell 12.")

## CELL 12 - All-Modes Pose Convergence Analysis (Top 10)

Parses all 9 Vina modes per top-10 compound.
Modes 1-3 within 0.5 kcal/mol + spread < 2.0 = HIGH confidence pose.

In [ ]:
import os, re, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

df_ranked = pd.read_csv("reports/final_ranked_compounds.csv")
top10_cids = df_ranked.head(10)["CID"].astype(str).tolist()
NAME_MAP = df_ranked.set_index("CID")["Compound"].astype(str).to_dict()

convergence_data = []
print("Pose Convergence Analysis (all 9 Vina modes):\n")
print(f"{'Compound':<30} {'M1':>7} {'M2':>7} {'M3':>7}  {'Spread':>7}  Confidence")
print("-"*75)

for cid in top10_cids:
    out_file = f"docking_results/{cid}_out.pdbqt"
    if not os.path.exists(out_file): continue
    with open(out_file) as f: content = f.read()
    affinities = []
    for line in content.split("\n"):
        if "VINA RESULT" in line:
            try: affinities.append(float(line.split()[3]))
            except (IndexError, ValueError): pass
    if not affinities: continue
    name = str(NAME_MAP.get(str(cid), cid))[:28]
    spread = max(affinities)-min(affinities)
    m1 = affinities[0] if len(affinities)>0 else None
    m2 = affinities[1] if len(affinities)>1 else None
    m3 = affinities[2] if len(affinities)>2 else None
    converged = bool(m1 and m2 and m3 and abs(m2-m1)<0.5 and abs(m3-m1)<0.5)
    confidence = "HIGH" if converged and spread<2.0 else ("MEDIUM" if spread<3.0 else "LOW")
    convergence_data.append({"CID":cid,"Compound":NAME_MAP.get(str(cid),cid),"Mode1":m1,"Mode2":m2,"Mode3":m3,
        "AllModes":affinities,"Spread_kcal":round(spread,3),"N_Modes":len(affinities),
        "Converged_Top3":converged,"Confidence":confidence})
    m2s = f"{m2:.3f}" if m2 else "N/A"; m3s = f"{m3:.3f}" if m3 else "N/A"
    print(f"  {name:<30} {m1:>7.3f} {m2s:>7} {m3s:>7}  {spread:>7.3f}  {confidence}")

if convergence_data:
    pd.DataFrame([{k:v for k,v in d.items() if k!="AllModes"} for d in convergence_data]).to_csv("reports/pose_convergence.csv", index=False)
    fig, ax = plt.subplots(figsize=(12,6))
    labels = [str(d["Compound"])[:22] for d in convergence_data]
    for i, d in enumerate(convergence_data):
        modes = d["AllModes"]
        ax.scatter([i]*len(modes), modes, color="#9ecae1", s=18, alpha=0.6, zorder=2)
        ax.scatter([i], [modes[0]], color="#08519c", s=75, zorder=3, marker="D", label="Mode 1 (best)" if i==0 else "")
    ax.set_xticks(range(len(convergence_data))); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
    ax.set_ylabel("Binding Affinity (kcal/mol)", fontsize=11, fontweight="bold")
    ax.set_title("Pose Convergence - All 9 Docking Modes (Top 10 Compounds)\nBlue diamond = Mode 1 | Light dots = Modes 2-9",
                 fontsize=11, fontweight="bold")
    ax.grid(axis="y", alpha=0.3, linestyle=":"); ax.legend(fontsize=9)
    plt.tight_layout()
    fig.savefig("reports/figures/fig5_pose_convergence.png", dpi=300, bbox_inches="tight")
    fig.savefig("reports/figures/fig5_pose_convergence.svg", bbox_inches="tight")
    plt.close()
    print("\nFigure saved: reports/figures/fig5_pose_convergence.png")
    print("Data saved:   reports/pose_convergence.csv")
print("\nProceed to Cell 13.")

## CELL 13 - PyMOL Visualization Script Generator

Generates individual .pml scripts for each top-10 compound:
- Transparent receptor surface + cartoon
- Ligand as magenta sticks
- Key CXCR4 residues shown (Asp171, Asp262, Glu288, Tyr116, His281, Trp94)
- H-bond distances auto-measured
- 2400x2400 PNG auto-rendered at publication quality

In [ ]:
import os, pandas as pd

df_ranked = pd.read_csv("reports/final_ranked_compounds.csv")
os.makedirs("pymol_scripts", exist_ok=True)
try:
    with open("receptor/receptor_path.txt") as f: receptor_path = f.read().strip()
except Exception:
    pdb_files = [f for f in os.listdir("receptor") if f.endswith(".pdb")]
    receptor_path = f"receptor/{pdb_files[0]}" if pdb_files else "receptor/receptor.pdb"

master_lines = ["# Master PyMOL script - top 10 compounds","bg_color white",
                "set ray_shadows, 0","set ambient, 0.5","set cartoon_fancy_helices, 1",""]

for _, row in df_ranked.head(10).iterrows():
    cid   = str(row["CID"])
    name  = str(row["Compound"]).replace(" ","_").replace("[","").replace("]","")
    score = row["Best_Affinity_kcal_mol"]
    rank  = int(row["Rank"])
    pose_pdb    = f"top10_poses/{cid}_top_pose.pdb"
    script_file = f"pymol_scripts/{rank:02d}_{name}_{cid}.pml"
    png_out     = f"pymol_scripts/{rank:02d}_{name}_{cid}.png"
    lines = [
        f"# Rank {rank} - {name} | CID {cid} | {score:.3f} kcal/mol | CXCR4 3ODU | Vina 1.2.5 exhaustiveness=32",
        "reinitialize","bg_color white","set ray_shadows, 0","set ambient, 0.5",
        "set spec_reflect, 0.2","set cartoon_fancy_helices, 1","set surface_quality, 1","",
        f"load {receptor_path}, receptor",f"load {pose_pdb}, ligand","",
        "hide everything, receptor","show cartoon, receptor","color gray80, receptor",
        "show surface, receptor","set transparency, 0.25, receptor","",
        "hide everything, ligand","show sticks, ligand","color magenta, ligand","set stick_radius, 0.15, ligand","",
        "# Key CXCR4 binding residues (3ODU numbering: Asp171, Asp262, Glu288, Tyr116, His281, Trp94, Asn192)",
        "select binding_site, receptor and (resi 171+262+288+116+281+94+192)",
        "show sticks, binding_site","color cyan, binding_site",
        "label binding_site and name CA, '%s%s' % (resn,resi)",
        "set label_size, 14","set label_color, black","",
        "# H-bond distances cutoff 3.5 Angstrom",
        "distance hbonds, ligand, binding_site, 3.5, mode=2",
        "color yellow, hbonds","hide labels, hbonds","",
        "zoom ligand, 8","orient ligand","",
        "set antialias, 2","set ray_trace_mode, 1","set ray_trace_gain, 0.1",
        f"ray 2400, 2400",f"png {png_out}, dpi=300",
        f"save pymol_scripts/{rank:02d}_{name}_{cid}.pse",
    ]
    with open(script_file,"w") as f: f.write("\n".join(lines))
    master_lines += [f"# Rank {rank}: {name} ({score:.3f} kcal/mol)", f"@{script_file}", ""]
    print(f"  Rank {rank:>2} | {str(name)[:30]:<30} -> {script_file}")

with open("pymol_scripts/00_RUN_ALL.pml","w") as f: f.write("\n".join(master_lines))
print(f"\nAll scripts saved to pymol_scripts/")
print("Usage: PyMOL -> File -> Run Script -> select .pml file")
print("Or in PyMOL command line: @pymol_scripts/01_CompoundName_CID.pml")
print("\nProceed to Cell 14.")

## CELL 14 - Auto-Generate Manuscript Methods Section

In [ ]:
import pandas as pd, os

df_ranked = pd.read_csv("reports/final_ranked_compounds.csv")
df_ctrl   = pd.read_csv("reports/control_compounds.csv")
n_compounds = len(df_ranked)
try:
    amd_score = df_ctrl[df_ctrl["Compound"].str.contains("AMD3100",na=False)].iloc[0]["Best_Affinity_kcal_mol"]
    amd_str = f"{amd_score:.3f}"
except Exception:
    amd_str = "reported"

top1 = df_ranked.iloc[0]

text = f"""
METHODS - Molecular Docking (paste into manuscript)
================================================================

Section X: Molecular Docking

The three-dimensional crystal structure of human CXCR4 in complex
with the small-molecule antagonist IT1t was obtained from the RCSB
Protein Data Bank (PDB ID: 3ODU, resolution: 2.5 Angstrom) [1].
Prior to docking, the receptor was prepared in PyMOL (Schrodinger
LLC): all water molecules, the co-crystallised ligand, and HETATM
records were removed, retaining only protein ATOM records. Polar
hydrogens were added and Gasteiger partial charges assigned using
OpenBabel (v3.1) [2] at physiological pH (7.4).

A library of {n_compounds} bioactive natural compounds was assembled
from PubChem Compound in SDF format with MMFF94-optimised 3D
coordinates. Structures were validated using RDKit [3] to confirm 3D
integrity, and physicochemical descriptors were computed (MW, HBD,
HBA, logP, TPSA, rotatable bonds). Lipinski Rule-of-Five compliance
was assessed and duplicate canonical SMILES were identified. SDF files
were converted to PDBQT format using OpenBabel at pH 7.4.

Molecular docking was performed using AutoDock Vina 1.2.5 [4]. The
search space was defined as a 30x30x30 Angstrom grid box centred on
the IT1t crystallographic binding pose (X=20.610, Y=-7.972,
Z=71.068 Angstrom), encompassing the extracellular vestibule formed
by TM3-TM7 and ECL2. Exhaustiveness was set to 32 with nine binding
modes per compound.

Validation used three reference compounds: AMD3100 (Plerixafor; CID
65015; docking score: {amd_str} kcal/mol) as positive control; IT1t
(3ODU co-crystal) as structural reference; and Pirfenidone (approved
IPF therapy) as negative reference. Hits were defined as compounds
outperforming AMD3100 (< {amd_str} kcal/mol).

Protein-ligand interaction profiles for the top 10 compounds were
generated using PLIP [5] characterising hydrogen bonds, hydrophobic
contacts, pi-stacking, and salt bridges. Pose reproducibility was
assessed by convergence analysis across all nine docking modes.

The highest-affinity compound was {top1["Compound"]}
(CID: {top1["CID"]}; {top1["Best_Affinity_kcal_mol"]:.3f} kcal/mol).

REFERENCES:
  [1] Wu B et al. (2010) Science 330:1066.
  [2] O'Boyle NM et al. (2011) J Cheminformatics 3:33.
  [3] RDKit: Open-source cheminformatics. https://www.rdkit.org
  [4] Eberhardt J et al. (2021) J Chem Inf Model 61:3891.
  [5] Salentin S et al. (2015) Nucleic Acids Res 43:W443.
================================================================
\"\"\"

print(text)
with open("reports/methods_section.txt","w") as f: f.write(text)
print("Saved: reports/methods_section.txt\nProceed to Cell 15.")

## CELL 15 - Package and Download All Results

In [ ]:
import zipfile, os
from google.colab import files

ZIP_NAME = "CXCR4_Docking_Publication_Results.zip"
print(f"Packaging all results...\n")

def add_dir(zf, folder, arc_prefix):
    if not os.path.isdir(folder): return
    for root, dirs, filenames in os.walk(folder):
        for fname in filenames:
            fpath = os.path.join(root, fname)
            arcname = os.path.join(arc_prefix, os.path.relpath(fpath, folder))
            zf.write(fpath, arcname)

with zipfile.ZipFile(ZIP_NAME, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in ["reports/final_ranked_compounds.csv","reports/full_results_with_ADMET.csv",
              "reports/control_compounds.csv","reports/docking_results_raw.csv",
              "reports/ligand_QC.csv","reports/conversion_report.csv",
              "reports/plip_interaction_summary.csv","reports/pose_convergence.csv",
              "reports/methods_section.txt"]:
        if os.path.exists(f): zf.write(f, f"tables/{os.path.basename(f)}")
    add_dir(zf, "reports/figures",  "figures")
    add_dir(zf, "top10_poses",      "poses_top10_pdb")
    add_dir(zf, "poses_pdb",        "poses_all_pdb")
    add_dir(zf, "plip_reports",     "plip_reports")
    add_dir(zf, "pymol_scripts",    "pymol_scripts")
    for f in os.listdir("docking_results"):
        if f.endswith("_out.pdbqt"):
            zf.write(f"docking_results/{f}", f"vina_output_pdbqt/{f}")

size_mb = os.path.getsize(ZIP_NAME)/(1024*1024)
print(f"ZIP size: {size_mb:.1f} MB")
print("\nContents:")
print("  tables/              - all CSV result tables + methods text")
print("  figures/             - 5 figures (PNG 300dpi + SVG vector)")
print("  poses_top10_pdb/     - top 10 best-pose PDB files")
print("  poses_all_pdb/       - all compound best-pose PDB files")
print("  plip_reports/        - PLIP XML + TXT interaction profiles")
print("  pymol_scripts/       - ready-to-run .pml scripts + .pse sessions")
print("  vina_output_pdbqt/   - raw Vina multi-mode PDBQT files")
print("\nDownloading...")
files.download(ZIP_NAME)
print("\nPIPELINE COMPLETE.")
print("1. Open tables/final_ranked_compounds.csv in Excel")
print("2. Load top10_poses/*.pdb in PyMOL")
print("3. Run pymol_scripts/00_RUN_ALL.pml for automated rendering")
print("4. Paste reports/methods_section.txt into your manuscript")